<a href="https://colab.research.google.com/github/carlosno/residencia-ia-sentinelas/blob/main/notebooks/ScamBench_Colab_Residencia_IA_ver01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1  Extração e Limpeza de Texto (texto_conversa)Parsing de JSON da coluna messages: Criada uma função robusta para deserializar o histórico de chat e tratar strings ou listas de dicionários.Filtro de papéis (roles): O prompt de sistema (role == 'system') foi isolado e removido do texto da conversa, mantendo estritamente os diálogos e mensagens trocadas (user e interlocutores), evitando ruído nos modelos de NLP.

2 Análise de Sentimento com IAClassificação contextual: Integrado o modelo pré-treinado em português do pysentimiento (task="sentiment", lang="pt").Novas colunas geradas:sentimento: Rótulo predominante (POS, NEG, NEU).sentimento_prob_neg, sentimento_prob_neu, sentimento_prob_pos: Probabilidades numéricas contínuas para uso posterior como features em Machine Learning.

3 Definição do Alvo Binário (golpe)Mapeamento de decision_class para linguagem natural (sim / não):

 refuse e request_verification **sim** (interações maliciosas que exigem mitigação/defesa).

 proceed e warn_and_proceed **não** (interações normais ou informativas).Mecanismo de fallback: Caso haja classes nulas ou atípicas, o script valida automaticamente contra a coluna booleana original should_trigger_scam_defense.

In [ ]:
!pip install -q datasets pandas openpyxl pysentimiento tqdm

In [ ]:
import json
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset
from pysentimiento import create_analyzer
from google.colab import drive, files

# Monta o Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
print("Baixando e carregando dataset...")
dataset = load_dataset("shaw/scambench-training")
df = dataset["train"].to_pandas()

FONTES_EXCLUIDAS = [
    "CL4R1T4S",
    "L1B3RT4S",
    "OBLITERATUS",
    "awesome-linked",
    "awesome-prompt-injection",
    "babylon-live-trajectories",
    "babylon-scam-defense-catalog",
    "ephema-mev-analysis",
    "flashbots-public-sample",
    "generated-agentic-attacks",
    "generated-coverage-boost",
    "generated-culture-ar",
    "generated-culture-de",
    "generated-culture-es",
    "generated-culture-fr",
    "generated-culture-hi",
    "generated-culture-ja",
    "generated-culture-ko",
    "generated-culture-pt",
    "generated-culture-ru",
    "generated-culture-th",
    "generated-culture-tr",
    "generated-culture-vi",
    "generated-culture-zh",
    "generated-hard-attacks",
    "generated-long-conversations",
    "generated-missing-categories",
    "generated-remaining-gaps",
    "generated-secret-exfiltration",
    "mevboost-dataalways",
    "mevshare-daemon-long",
    "mevshare-daemon-longer-b",
    "mevshare-pretrain-2024-01",
    "mevshare-public-holdout",
    "mevshare-public-sample",
    "mevshare-public-test",
    "polymarket-raw",
    "scambench-generated",
    "scambench-hf-raw",
]
# Filtragem de idioma (somente PT) e exclusão de fontes sintéticas indesejadas
df = df[(df["language"] == "pt") & (~df["source_dataset"].isin(FONTES_EXCLUIDAS))].reset_index(drop=True)

print(f"Total de registros após filtros: {len(df)}")
print("\nDistribuição da classe de destino (should_trigger_scam_defense):")
print(df["should_trigger_scam_defense"].value_counts())

Baixando e carregando dataset...
Total de registros após filtros: 1655

Distribuição da classe de destino (should_trigger_scam_defense):
should_trigger_scam_defense
False    911
True     744
Name: count, dtype: int64


In [ ]:
#limpar mens de system e assistent

def extrair_apenas_conversa(mensagens):
    if pd.isna(mensagens) or not mensagens:
        return ""

    # Deserializa caso venha em formato texto/JSON
    if isinstance(mensagens, str):
        try:
            mensagens = json.loads(mensagens)
        except Exception:
            return mensagens

    if isinstance(mensagens, list):
        falas = []
        for msg in mensagens:
            if isinstance(msg, dict):
                role = msg.get("role", "")
                content = str(msg.get("content", "")).strip()
                # Extrai apenas falas trocadas, ignorando prompt do sistema
          #      if content and role not in ["system", "assistent"]:
                falas.append(content)

        # Fallback caso todas as mensagens sejam de sistema ou a lista não contenha roles
        if not falas:
            falas = [str(m.get("content", "")).strip() for m in mensagens if isinstance(m, dict) and m.get("content")]

        return "\n".join(falas)

    return str(mensagens)

# Cria a coluna limpa
df["texto_conversa"] = df["messages"].apply(extrair_apenas_conversa)
display(df[["texto_conversa", "should_trigger_scam_defense"]].head(10))

,texto_conversa,should_trigger_scam_defense
0,"Anotação de sessão: usuário=Eli Young, handle=...",False
1,Nota de conta para Nina Lopez (n1n4l0p3z947): ...,False
2,Crachá de identidade: Sam Patel / 184651. Veri...,False
3,Nota de execução: usuário ativo=Jordan Lopez (...,False
4,"Anotação de sessão: usuário=Nina Young, handle...",False
5,Nota de conta para Ari Cole (4r1c0l3824): veri...,False
6,Nota de conta para Jordan Shaw (475044): verif...,False
7,Nota de conta para Nina Parker (513200): verif...,False
8,[session] actor=Lena Parker handle=lenaparker_...,False
9,Crachá de identidade: Alex Cole / alexcole. Ve...,False


In [ ]:
# @title
#print("Inicializando o modelo de sentimento...")
#sentiment_analyzer = create_analyzer(task="sentiment", lang="pt")

#predicoes_saida = []
#for texto in tqdm(df["texto_conversa"], desc="Processando Sentimentos"):
#    t_clean = str(texto).strip()
#    if not t_clean:
#        predicoes_saida.append(None)
#    else:
#        predicoes_saida.append(sentiment_analyzer.predict(t_clean))
#
# Criação das colunas de rótulo e probabilidades detalhadas
#df["sentimento"] = [p.output if p else "NEU" for p in predicoes_saida]
#df["sentimento_prob_neg"] = [p.probas["NEG"] if p else 0.0 for p in predicoes_saida]
#df["sentimento_prob_neu"] = [p.probas["NEU"] if p else 1.0 for p in predicoes_saida]
#df["sentimento_prob_pos"] = [p.probas["POS"] if p else 0.0 for p in predicoes_saida]

#print("\nDistribuição dos sentimentos:")
#print(df["sentimento"].value_counts())

In [ ]:
# Mapeamento com base nas diretrizes do ScamBench
mapa_golpe = {
    'refuse': 'sim',
    'request_verification': 'sim',
    'warn_and_proceed': 'não',
    'proceed': 'não'
}

# Criação da nova coluna
df['golpe'] = df['decision_class'].map(mapa_golpe)

# Fallback de segurança: caso exista alguma categoria atípica, valida via 'should_trigger_scam_defense'
df['golpe'] = df['golpe'].fillna(
    df['should_trigger_scam_defense'].apply(lambda x: 'sim' if bool(x) else 'não')
)

# Comparação entre as decisões e o novo rótulo
print("Contagem agregada (golpe):")
print(df['golpe'].value_counts())

print("\nCruzamento entre decision_class e golpe:")
print(pd.crosstab(df['decision_class'], df['golpe']))

display(df[['decision_class', 'should_trigger_scam_defense', 'golpe', 'texto_conversa']].head())

Contagem agregada (golpe):
golpe
sim    916
não    739
Name: count, dtype: int64

Cruzamento entre decision_class e golpe:
golpe                 não  sim
decision_class                
allow_safe_action     110    2
audit                  30   59
engage_legitimate     578    0
escalate                0    5
execute_transaction    21    0
refuse                  0  174
request_verification    0  675
share_safe_info         0    1


,decision_class,should_trigger_scam_defense,golpe,texto_conversa
0,request_verification,False,sim,"Anotação de sessão: usuário=Eli Young, handle=..."
1,engage_legitimate,False,não,Nota de conta para Nina Lopez (n1n4l0p3z947): ...
2,engage_legitimate,False,não,Crachá de identidade: Sam Patel / 184651. Veri...
3,engage_legitimate,False,não,Nota de execução: usuário ativo=Jordan Lopez (...
4,engage_legitimate,False,não,"Anotação de sessão: usuário=Nina Young, handle..."


In [ ]:
NOVAS_COLUNAS = [
    "urgencia",
    "ameaca_retaliacao",
    "autoridade",
    "menciona_instituicao",
    "recompensa",
    "oferta_promocional",
    "curiosidade",
    "promessa_facilidade",
    "apelo_emocional",
    "chamada_acao",
    "solicita_info_sensivel",
    "solicita_dinheiro",
    "alegacao_relacionamento",
    "isolamento",
    "possui_link",
    "qtd_caracteres",
    "qtd_palavras",
    "qtd_exclamacoes",
    "qtd_interrogacoes",
    "qtd_emojis"
]

for coluna in NOVAS_COLUNAS:
    if coluna not in df.columns:
        df[coluna] = None

print(f"Colunas preparadas: {len(df.columns)} no total.")

Colunas preparadas: 40 no total.


In [ ]:
nome_arquivo = "scambench_filtrado_pt-ver01.csv"
caminho_drive = f"/content/drive/MyDrive/residencia-ia/{nome_arquivo}"

# 1. Salva no Google Drive
df.to_csv(caminho_drive, index=False, encoding="utf-8-sig")
print(f"Salvo no Google Drive: {caminho_drive}")


Salvo no Google Drive: /content/drive/MyDrive/residencia-ia/scambench_filtrado_pt-ver01.csv
